## Importing Libraries and Loading our Models.

In this section, we import all required libraries and load the pretrained Stack and Kaggle BERT models along with their corresponding tokenizers.

In [1]:
import torch
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from utils import tokenize

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model_path_stack = "nemoonyte/stack-bert-model"
model_path_kaggle = "nemoonyte/kaggle-bert-model"

tokenizer_stack = BertTokenizer.from_pretrained(model_path_stack)
tokenizer_kaggle = BertTokenizer.from_pretrained(model_path_kaggle)

model_stack = BertForSequenceClassification.from_pretrained(model_path_stack).to(device)
model_kaggle = BertForSequenceClassification.from_pretrained(model_path_kaggle).to(device)

model_stack.eval()
model_kaggle.eval()

cpu


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

## Loading and Inspect Test Datasets

We load both the Stack Overflow and Kaggle test datasets and verify their structure.

In [2]:

# now we load both test datasets

test_df_stack = pd.read_csv("dataset/stack_test_clean.csv")
test_df_kaggle = pd.read_csv("dataset/kaggle_test_clean.csv")

print("Stack:", test_df_stack.shape)
print(test_df_stack.columns.tolist())

print("\nKaggle:", test_df_kaggle.shape)
print(test_df_kaggle.columns.tolist())

Stack: (1600, 3)
['source', 'raw_text', 'priority_label']

Kaggle: (1000, 3)
['source', 'raw_text', 'priority_label']


## Defining Dataset Class and Evaluation Function

We define a PyTorch dataset class and an evaluation function to run inference and compute predictions.

In [3]:
label_map = {
    "Low": 0,
    "Medium": 1,
    "High": 2}

text_column = "raw_text"
label_column = "priority_label"

class QueryDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.reset_index(drop=True)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels.iloc[idx], dtype=torch.long)
        return item

def evaluate_model(model, dataloader):
    all_preds, all_true = [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().tolist())
            all_true.extend(labels.cpu().tolist())

    return all_true, all_preds

print(label_map)

{'Low': 0, 'Medium': 1, 'High': 2}


## Evaluating Kaggle Model on Kaggle Test Set

This evaluates how well the Kaggle-trained model performs on its own dataset.

In [4]:
# now we test the kaggle model on the kaggle test set

# first we tokenize the text so the model can understand it


kaggle_encodings, kaggle_labels = tokenize(
    test_df_kaggle,
    text_column=text_column,
    tokenizer=tokenizer_kaggle,
    labels=label_column
)
# convert labels to numbers (same as training)
kaggle_labels_encoded = kaggle_labels.map(label_map)

kaggle_loader = DataLoader(
    QueryDataset(kaggle_encodings, kaggle_labels_encoded),
    batch_size=16,
    shuffle=False
)

kaggle_true, kaggle_preds = evaluate_model(model_kaggle, kaggle_loader)

print("Kaggle Model → Kaggle Test Set")
print("Accuracy:", accuracy_score(kaggle_true, kaggle_preds))
print(classification_report(kaggle_true, kaggle_preds, target_names=["Low","Medium","High"]))
print(confusion_matrix(kaggle_true, kaggle_preds))

Kaggle Model → Kaggle Test Set
Accuracy: 1.0
              precision    recall  f1-score   support

         Low       1.00      1.00      1.00       360
      Medium       1.00      1.00      1.00       311
        High       1.00      1.00      1.00       329

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000

[[360   0   0]
 [  0 311   0]
 [  0   0 329]]


## Evaluating Kaggle Model on Stack Test Set

This tests how well the Kaggle model generalizes to Stack Overflow data.

In [5]:
stack_enc_kaggle, stack_lab_kaggle = tokenize(
    test_df_stack,
    text_column=text_column,
    tokenizer=tokenizer_kaggle,
    labels=label_column
)

stack_lab_kaggle_encoded = stack_lab_kaggle.map(label_map)

stack_loader_kaggle = DataLoader(
    QueryDataset(stack_enc_kaggle, stack_lab_kaggle_encoded),
    batch_size=16,
    shuffle=False
)

kaggle_on_stack_true, kaggle_on_stack_preds = evaluate_model(model_kaggle, stack_loader_kaggle)

print("Kaggle Model → Stack Test Set")
print("Accuracy:", accuracy_score(kaggle_on_stack_true, kaggle_on_stack_preds))

Kaggle Model → Stack Test Set
Accuracy: 0.48375


## Evaluating Stack Model on Stack Test Set

This evaluates how well the Stack-trained model performs on its own dataset.

In [6]:
stack_encodings, stack_labels = tokenize(
    test_df_stack,
    text_column=text_column,
    tokenizer=tokenizer_stack,
    labels=label_column
)

stack_labels_encoded = stack_labels.map(label_map)

stack_loader = DataLoader(
    QueryDataset(stack_encodings, stack_labels_encoded),
    batch_size=16,
    shuffle=False
)

stack_true, stack_preds = evaluate_model(model_stack, stack_loader)

print("Stack Model → Stack Test Set")
print("Accuracy:", accuracy_score(stack_true, stack_preds))
print(classification_report(stack_true, stack_preds, target_names=["Low","Medium","High"]))
print(confusion_matrix(stack_true, stack_preds))

Stack Model → Stack Test Set
Accuracy: 0.921875
              precision    recall  f1-score   support

         Low       0.87      0.97      0.92       611
      Medium       0.96      0.93      0.95       923
        High       1.00      0.29      0.45        66

    accuracy                           0.92      1600
   macro avg       0.94      0.73      0.77      1600
weighted avg       0.93      0.92      0.92      1600

[[595  16   0]
 [ 62 861   0]
 [ 25  22  19]]


## Evaluating Stack Model on Kaggle Test Set

This tests how well the Stack model generalizes to Kaggle data.

In [7]:
kaggle_enc_stack, kaggle_lab_stack = tokenize(
    test_df_kaggle,
    text_column=text_column,
    tokenizer=tokenizer_stack,
    labels=label_column
)

kaggle_lab_stack_encoded = kaggle_lab_stack.map(label_map)

kaggle_loader_stack = DataLoader(
    QueryDataset(kaggle_enc_stack, kaggle_lab_stack_encoded),
    batch_size=16,
    shuffle=False
)

stack_on_kaggle_true, stack_on_kaggle_preds = evaluate_model(model_stack, kaggle_loader_stack)

print("Stack Model → Kaggle Test Set")
print("Accuracy:", accuracy_score(stack_on_kaggle_true, stack_on_kaggle_preds))

Stack Model → Kaggle Test Set
Accuracy: 0.35


## Loading and Inspecting the Twitter Dataset

We load the labeled Twitter dataset contributed by the team and verify its structure before evaluation.

In [12]:
test_df_twitter = pd.read_csv("updated.tweets.labeled.csv")

# Strip whitespace from all column names
test_df_twitter.columns = test_df_twitter.columns.str.strip()

# The Twitter CSV uses "text" and "Priority" instead of "raw_text" and "priority_label"
test_df_twitter = test_df_twitter.rename(columns={"text": "raw_text", "Priority": "priority_label"})

# Normalize labels: replace non-breaking spaces (\xa0) and any other whitespace,
# then title-case to fix inconsistent casing (e.g. "low" → "Low")
test_df_twitter["priority_label"] = (
    test_df_twitter["priority_label"]
    .str.replace(u'\xa0', ' ', regex=False)   # non-breaking space
    .str.replace(r'\s+', ' ', regex=True)      # collapse any whitespace
    .str.strip()
    .str.title()                                # "low" → "Low", "medium" → "Medium"
)

test_df_twitter = test_df_twitter[test_df_twitter["priority_label"].isin(["Low", "Medium", "High"])].reset_index(drop=True)

print("Twitter:", test_df_twitter.shape)
print(test_df_twitter.columns.tolist())
print("\nLabel distribution:")
print(test_df_twitter["priority_label"].value_counts())
test_df_twitter.head()

Twitter: (247, 10)
['id', 'date', 'author_id', 'raw_text', 'like_count', 'reply_count', 'retweet_count', 'quote_count', 'company', 'priority_label']

Label distribution:
priority_label
Low       192
Medium     38
High       17
Name: count, dtype: int64


,id,date,author_id,raw_text,like_count,reply_count,retweet_count,quote_count,company,priority_label
0,2.041800e+18,2026-04-08T08:47:41.000Z,1.588360e+18,@ToonHive Comcast: (gasp ) what are we gonna d...,0,0,0,0,Comcast,Low
1,2.041800e+18,2026-04-08T08:38:11.000Z,4.831046e+08,@XfinitySupport @Xfinity I have closed my acc...,0,1,0,0,Comcast,High
2,2.041800e+18,2026-04-08T08:34:20.000Z,1.490120e+18,"""Comcast doesn't care about you."" Truer words ...",0,0,0,0,Comcast,Low
3,2.041790e+18,2026-04-08T08:19:09.000Z,5.308978e+07,Okay @XfinitySupport @Xfinity why is it when I...,0,1,0,0,Comcast,Low
4,2.041790e+18,2026-04-08T08:11:01.000Z,1.040002e+08,Netflix went up AGAIN!!! Might as well just ge...,0,0,0,0,Comcast,Low


## Evaluating Stack Model on Twitter Dataset

This tests how well the Stack Overflow-trained model generalizes to real-world Twitter/X customer complaint data.

In [15]:
twitter_enc_stack, twitter_lab_stack = tokenize(
    test_df_twitter,
    text_column=text_column,
    tokenizer=tokenizer_stack,
    labels=label_column
)

twitter_lab_stack_encoded = twitter_lab_stack.map(label_map)


twitter_lab_stack_encoded = twitter_lab_stack_encoded.astype(int).reset_index(drop=True)

twitter_loader_stack = DataLoader(
    QueryDataset(twitter_enc_stack, twitter_lab_stack_encoded),
    batch_size=16,
    shuffle=False
)

stack_on_twitter_true, stack_on_twitter_preds = evaluate_model(model_stack, twitter_loader_stack)

print("Stack Model → Twitter Dataset")
print("Accuracy:", accuracy_score(stack_on_twitter_true, stack_on_twitter_preds))
print(classification_report(stack_on_twitter_true, stack_on_twitter_preds, target_names=["Low", "Medium", "High"]))
print(confusion_matrix(stack_on_twitter_true, stack_on_twitter_preds))

Stack Model → Twitter Dataset
Accuracy: 0.7449392712550608
              precision    recall  f1-score   support

         Low       0.78      0.95      0.86       192
      Medium       0.00      0.00      0.00        38
        High       0.08      0.06      0.07        17

    accuracy                           0.74       247
   macro avg       0.29      0.34      0.31       247
weighted avg       0.61      0.74      0.67       247

[[183   0   9]
 [ 36   0   2]
 [ 16   0   1]]


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Evaluating Kaggle Model on Twitter Dataset

This tests how well the Kaggle-trained model generalizes to real-world Twitter/X customer complaint data.

In [17]:
twitter_enc_kaggle, twitter_lab_kaggle = tokenize(
    test_df_twitter,
    text_column=text_column,
    tokenizer=tokenizer_kaggle,
    labels=label_column
)

twitter_lab_kaggle_encoded = twitter_lab_kaggle.map(label_map)



twitter_lab_kaggle_encoded = twitter_lab_kaggle_encoded.astype(int).reset_index(drop=True)

twitter_loader_kaggle = DataLoader(
    QueryDataset(twitter_enc_kaggle, twitter_lab_kaggle_encoded),
    batch_size=16,
    shuffle=False
)

kaggle_on_twitter_true, kaggle_on_twitter_preds = evaluate_model(model_kaggle, twitter_loader_kaggle)

print("Kaggle Model → Twitter Dataset")
print("Accuracy:", accuracy_score(kaggle_on_twitter_true, kaggle_on_twitter_preds))
print(classification_report(kaggle_on_twitter_true, kaggle_on_twitter_preds, target_names=["Low", "Medium", "High"]))
print(confusion_matrix(kaggle_on_twitter_true, kaggle_on_twitter_preds))

Kaggle Model → Twitter Dataset
Accuracy: 0.20647773279352227
              precision    recall  f1-score   support

         Low       0.89      0.04      0.08       192
      Medium       0.16      0.92      0.27        38
        High       0.42      0.47      0.44        17

    accuracy                           0.21       247
   macro avg       0.49      0.48      0.27       247
weighted avg       0.74      0.21      0.13       247

[[  8 176   8]
 [  0  35   3]
 [  1   8   8]]
